# Compare mTAN Optical Encoders: ALBEF vs Optical-Only

This notebook compares the optical time-encoding branch of two trained models:

- `ALBEF` checkpoint: `<BASE_DIR>/data/model/checkpoints_bns_nsbh/bns_nsbh_v7_win1020/ALBEF/albef_best.pth`
- `Optical-only` checkpoint: `<BASE_DIR>/data/model/checkpoints_optical/optical_only/optical_only_kn_v15_win1020/optical_only_best.pth`

The notebook produces two main diagnostics:

1. all-head, all-dimension mTAN time-embedding period heatmaps for both optical encoders;
2. time-attention heatmaps on the same training KN light curve, so differences come from the encoder rather than from different samples.

The default light-curve example reuses the known-good training sample from the existing optical-only attention example metadata.


In [ ]:
from __future__ import annotations

import importlib.util
import json
import sys
from contextlib import nullcontext
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from matplotlib.lines import Line2D
from torch.amp import autocast

plt.style.use('default')
torch.set_grad_enabled(False)
pd.set_option('display.max_columns', 200)

ROOT = Path('<BASE_DIR>/gw-kn-multimodal')
MODEL_PY = ROOT / 'Model' / 'model.py'
OUTPUT_DIR = ROOT / 'figures' / 'mtan_optical_encoder_comparison'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ALBEF_CKPT = Path('<BASE_DIR>/data/model/checkpoints_bns_nsbh/bns_nsbh_v8/ALBEF/albef_best.pth')
OPTICAL_ONLY_CKPT = Path('<BASE_DIR>/data/model/checkpoints_optical/optical_only/optical_only_kn_v16/optical_only_best.pth')
TRAIN_H5 = Path('<BASE_DIR>/data/Optical_Only_dataset/combined_dataset_train.h5')
OFFSET_NPZ = Path('<BASE_DIR>/data/Optical_Only_dataset/delta_days_distribution.npz')
OFFSET_KEY = 'delta_days_combined'
OFFSET_QUANTILES = (0.1, 0.3, 0.5, 0.7, 0.9)
EXAMPLE_SAMPLE_INDEX = 7450970

TIME_SCALE_DAYS = 100.0
PSFFLUX_ZP = 31.4
ASINH_MAG_FACTOR = 2.5 / np.log(10.0)
PERIOD_EPS = 1.0e-12
BANDS = ('u', 'g', 'r', 'i', 'z', 'y')
BAND_COLORS = {
    'u': '#6d6d6d',
    'g': '#2ca02c',
    'r': '#d62728',
    'i': '#ff7f0e',
    'z': '#8c564b',
    'y': '#9467bd',
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print({
    'device': str(device),
    'output_dir': str(OUTPUT_DIR),
    'albef_ckpt_exists': ALBEF_CKPT.exists(),
    'optical_only_ckpt_exists': OPTICAL_ONLY_CKPT.exists(),
    'train_h5_exists': TRAIN_H5.exists(),
    'offset_npz_exists': OFFSET_NPZ.exists(),
})


In [ ]:
def load_module(name: str, path: Path):
    if str(path.parent) not in sys.path:
        sys.path.insert(0, str(path.parent))
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    assert spec.loader is not None
    spec.loader.exec_module(mod)
    return mod


def namespace_to_dict(obj) -> dict:
    if isinstance(obj, dict):
        return dict(obj)
    if hasattr(obj, '__dict__'):
        return dict(vars(obj))
    return {}


def parse_mtan_lupt_m5(value):
    default = (23.9, 25.0, 24.7, 24.0, 23.3, 22.1)
    if value is None:
        return default
    if isinstance(value, str):
        pieces = [x.strip() for x in value.split(',') if x.strip()]
        if len(pieces) != 6:
            return default
        return tuple(float(x) for x in pieces)
    arr = np.asarray(value, dtype=np.float64).reshape(-1)
    if arr.size != 6:
        return default
    return tuple(float(x) for x in arr.tolist())


def clean_state_dict(state_dict: dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
    return {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}


def build_ref_time(batch_size: int, n_ref: int, ref_start: float, ref_end: float, device: torch.device, dtype: torch.dtype):
    ref = torch.linspace(float(ref_start), float(ref_end), int(n_ref), dtype=dtype, device=device)
    return ref.unsqueeze(0).repeat(batch_size, 1)


def apply_time_offsets(opt_t: torch.Tensor, opt_mask: torch.Tensor, delta_days: torch.Tensor, scale_divisor: float = TIME_SCALE_DAYS) -> torch.Tensor:
    shift = (delta_days.to(device=opt_t.device, dtype=opt_t.dtype) / float(scale_divisor)).unsqueeze(1)
    valid_slots = (opt_mask.sum(dim=-1) > 0).to(dtype=opt_t.dtype)
    return opt_t + shift * valid_slots


def load_offset_days(npz_path: Path, key: str = OFFSET_KEY, quantiles = OFFSET_QUANTILES) -> np.ndarray:
    with np.load(npz_path, allow_pickle=False) as npz:
        values = np.asarray(npz[key], dtype=np.float64).reshape(-1)
    values = values[np.isfinite(values)]
    if values.size == 0:
        raise ValueError(f'No finite values found in {npz_path}::{key}')
    return np.quantile(values, np.asarray(quantiles, dtype=np.float64))


def slugify(name: str) -> str:
    return name.lower().replace('-', '_').replace(' ', '_')


model_mod = load_module('gw_kn_model_compare_notebook', MODEL_PY)


def load_albef_bundle(ckpt_path: Path, device: torch.device) -> dict:
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    saved_args = namespace_to_dict(ckpt.get('args', {}))
    state_dict = clean_state_dict(ckpt['model_state_dict'])
    state_dict = model_mod.migrate_time_embed_state_dict(state_dict)
    fusion_mode = model_mod.normalize_fusion_mode(
        saved_args.get('fusion_mode', None),
        dual_fusion=bool(saved_args.get('dual_fusion', False)),
    )
    model = model_mod.GWOpticalALBEFModel(
        enc_dim=int(saved_args.get('enc_dim', 128)),
        proj_dim=int(saved_args.get('proj_dim', 256)),
        ref_time_dim=int(saved_args.get('ref_dim', 64)),
        use_lightweight_gw=bool(saved_args.get('use_lightweight_gw', False)),
        gw_dropout=float(saved_args.get('gw_dropout', 0.0)),
        opt_dropout=float(saved_args.get('opt_dropout', 0.0)),
        proj_dropout=float(saved_args.get('proj_dropout', 0.0)),
        fusion_dropout=float(saved_args.get('fusion_dropout', 0.0)),
        feature_dropout=float(saved_args.get('feature_dropout', 0.0)),
        label_smoothing=float(saved_args.get('label_smoothing', 0.0)),
        temp_init=float(saved_args.get('temp_init', 0.07)),
        temp_min=float(saved_args.get('temp_min', 0.01)),
        temp_max=float(saved_args.get('temp_max', 100.0)),
        itc_label_smoothing=float(saved_args.get('itc_label_smoothing', 0.0)),
        fusion_attn_dim=saved_args.get('fusion_attn_dim', None),
        fusion_hidden_dim=saved_args.get('fusion_hidden_dim', None),
        dual_fusion=bool(saved_args.get('dual_fusion', False)),
        fusion_mode=fusion_mode,
        use_similarity_as_cls_input=bool(saved_args.get('use_similarity_as_cls_input', False)),
        use_cred_level_feature=bool(saved_args.get('use_cred_level_feature', False)),
        time_compat_weight=float(saved_args.get('time_compat_weight', 0.6)),
        time_compat_tau_days=float(saved_args.get('time_compat_tau_days', 30.0)),
        time_compat_power=float(saved_args.get('time_compat_power', 2.0)),
        time_compat_max_penalty=float(saved_args.get('time_compat_max_penalty', 8.0)),
        mtan_snr_s0=float(saved_args.get('mtan_snr_s0', 3.0)),
        mtan_snr_beta=float(saved_args.get('mtan_snr_beta', 1.0)),
        mtan_snr_clip_min=float(saved_args.get('mtan_snr_clip_min', -8.0)),
        mtan_snr_clip_max=float(saved_args.get('mtan_snr_clip_max', 20.0)),
        mtan_snr_eps=float(saved_args.get('mtan_snr_eps', 1e-9)),
        mtan_lupt_psfflux_zp=float(saved_args.get('mtan_lupt_psfflux_zp', PSFFLUX_ZP)),
        mtan_lupt_k=float(saved_args.get('mtan_lupt_k', 1.0)),
        mtan_lupt_m5_mag=parse_mtan_lupt_m5(saved_args.get('mtan_lupt_m5_mag')),
    ).to(device)
    model.load_state_dict(state_dict, strict=True)
    model.eval()
    curve_encoder = model.optical_encoder.curve_encoder if hasattr(model.optical_encoder, 'curve_encoder') else model.optical_encoder
    return {
        'name': 'ALBEF',
        'kind': 'albef',
        'ckpt_path': ckpt_path,
        'ckpt': ckpt,
        'saved_args': saved_args,
        'state_dict': state_dict,
        'model': model,
        'curve_encoder': curve_encoder,
        'n_ref': int(saved_args.get('n_ref', 64)),
        'ref_start': float(saved_args.get('ref_start', -0.3)),
        'ref_end': float(saved_args.get('ref_end', 0.6)),
        'offset_scale_days_divisor': float(saved_args.get('neg_offset_scale_days_divisor', TIME_SCALE_DAYS)),
        'epoch': ckpt.get('epoch', None),
    }


def load_optical_only_bundle(ckpt_path: Path, device: torch.device) -> dict:
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    saved_args = namespace_to_dict(ckpt.get('args', {}))
    state_dict = clean_state_dict(ckpt['model_state_dict'])
    state_dict = model_mod.migrate_time_embed_state_dict(state_dict)
    has_universal_aux = (
        bool(saved_args.get('universal_train_enable', False))
        or any(key.startswith('projection_head.') for key in state_dict)
        or any(key.startswith('adv_head_n_det.') for key in state_dict)
    )
    head_hidden_dim = saved_args.get('head_hidden_dim', None)
    enc_dim = int(saved_args.get('enc_dim', 128))
    model = model_mod.OpticalKNClassifier(
        optical_input_dim=6,
        ref_time_dim=int(saved_args.get('ref_dim', 64)),
        enc_dim=enc_dim,
        num_heads=int(saved_args.get('num_heads', 4)),
        k_dim=int(saved_args.get('k_dim', 64)),
        opt_dropout=float(saved_args.get('opt_dropout', 0.1)),
        feature_dropout=float(saved_args.get('feature_dropout', 0.05)),
        head_hidden_dim=head_hidden_dim,
        head_dropout=float(saved_args.get('head_dropout', 0.2)),
        universal_aux_enable=bool(has_universal_aux),
        proj_dim=64,
        adv_hidden_dim=int(head_hidden_dim) if head_hidden_dim is not None else enc_dim,
        n_det_bucket_classes=5,
        n_bands_bucket_classes=4,
        t_span_bucket_classes=5,
        grl_lambda=float(saved_args.get('grl_lambda', 1.0)),
    ).to(device)
    model.load_state_dict(state_dict, strict=True)
    model.eval()
    return {
        'name': 'Optical-only',
        'kind': 'optical_only',
        'ckpt_path': ckpt_path,
        'ckpt': ckpt,
        'saved_args': saved_args,
        'state_dict': state_dict,
        'model': model,
        'curve_encoder': model.optical_encoder,
        'n_ref': int(saved_args.get('n_ref', 64)),
        'ref_start': float(saved_args.get('ref_start', -0.3)),
        'ref_end': float(saved_args.get('ref_end', 0.6)),
        'offset_scale_days_divisor': float(saved_args.get('offset_scale_days_divisor', TIME_SCALE_DAYS)),
        'epoch': ckpt.get('epoch', None),
    }


ALBEF = load_albef_bundle(ALBEF_CKPT, device)
OPTICAL_ONLY = load_optical_only_bundle(OPTICAL_ONLY_CKPT, device)
MODEL_BUNDLES = {
    ALBEF['name']: ALBEF,
    OPTICAL_ONLY['name']: OPTICAL_ONLY,
}

COMMON_WINDOW_SCALED = (
    max(bundle['ref_start'] for bundle in MODEL_BUNDLES.values()),
    min(bundle['ref_end'] for bundle in MODEL_BUNDLES.values()),
)
if COMMON_WINDOW_SCALED[0] >= COMMON_WINDOW_SCALED[1]:
    raise ValueError(f'No overlapping time window across models: {COMMON_WINDOW_SCALED}')

summary_df = pd.DataFrame([
    {
        'model': name,
        'epoch': bundle['epoch'],
        'n_ref': bundle['n_ref'],
        'ref_start': bundle['ref_start'],
        'ref_end': bundle['ref_end'],
        'offset_scale_days_divisor': bundle['offset_scale_days_divisor'],
        'checkpoint': str(bundle['ckpt_path']),
    }
    for name, bundle in MODEL_BUNDLES.items()
])
display(summary_df)
print({'common_window_scaled': COMMON_WINDOW_SCALED, 'common_window_days': tuple(x * TIME_SCALE_DAYS for x in COMMON_WINDOW_SCALED)})


## Period Heatmaps

The next cell extracts the learnable periodic embedding parameters from each checkpoint and draws a shared-scale heatmap of `log10(period [days])` for every head and periodic embedding dimension.


In [ ]:
def extract_time_embedding_arrays(bundle: dict):
    prefixes = [
        'optical_encoder.curve_encoder.time_embedding.',
        'optical_encoder.time_embedding.',
    ]
    state = bundle['state_dict']
    matched_prefix = None
    for prefix in prefixes:
        if prefix + 'log_wi' in state:
            matched_prefix = prefix
            break
    if matched_prefix is None:
        raise KeyError(f'No time_embedding.* keys found for {bundle["name"]}')

    w0 = state[matched_prefix + 'w0'].detach().cpu().numpy().reshape(-1, 1)
    a0 = state[matched_prefix + 'a0'].detach().cpu().numpy().reshape(-1, 1)
    wi = np.exp(state[matched_prefix + 'log_wi'].detach().cpu().numpy()).reshape(w0.shape[0], -1)
    ai = state[matched_prefix + 'ai'].detach().cpu().numpy().reshape(w0.shape[0], -1)
    return matched_prefix, w0, a0, wi, ai


def compute_period_result(bundle: dict) -> dict:
    prefix, w0, a0, wi, ai = extract_time_embedding_arrays(bundle)
    num_heads = int(wi.shape[0])
    num_dims = int(wi.shape[1] + 1)
    abs_wi = np.abs(wi)

    omega_scaled = np.full((num_heads, num_dims), np.nan, dtype=np.float64)
    period_scaled = np.full((num_heads, num_dims), np.nan, dtype=np.float64)
    period_days = np.full((num_heads, num_dims), np.nan, dtype=np.float64)
    omega_scaled[:, 1:] = abs_wi
    period_scaled[:, 1:] = np.where(abs_wi > PERIOD_EPS, 2.0 * np.pi / abs_wi, np.nan)
    period_days[:, 1:] = period_scaled[:, 1:] * TIME_SCALE_DAYS

    long_df = pd.DataFrame({
        'head': np.repeat(np.arange(num_heads, dtype=np.int64), num_dims),
        'dim': np.tile(np.arange(num_dims, dtype=np.int64), num_heads),
        'component': np.where(np.tile(np.arange(num_dims), num_heads) == 0, 'linear', 'periodic'),
        'omega_scaled': omega_scaled.reshape(-1),
        'period_scaled': period_scaled.reshape(-1),
        'period_days': period_days.reshape(-1),
    })
    long_df['dim_name'] = long_df['dim'].map(lambda d: f'dim_{d:03d}')

    matrix_df = pd.DataFrame(
        period_days,
        index=[f'head_{head_idx:02d}' for head_idx in range(num_heads)],
        columns=[f'dim_{dim_idx:03d}' for dim_idx in range(num_dims)],
    )
    return {
        'prefix': prefix,
        'w0': w0,
        'a0': a0,
        'wi': wi,
        'ai': ai,
        'omega_scaled': omega_scaled,
        'period_scaled': period_scaled,
        'period_days': period_days,
        'period_long_df': long_df,
        'period_matrix_df': matrix_df,
    }


def centers_to_edges(centers: np.ndarray, default_half_width: float = 0.5) -> np.ndarray:
    centers = np.asarray(centers, dtype=np.float64)
    if centers.size == 1:
        return np.asarray([centers[0] - default_half_width, centers[0] + default_half_width], dtype=np.float64)
    mids = 0.5 * (centers[:-1] + centers[1:])
    first = centers[0] - (mids[0] - centers[0])
    last = centers[-1] + (centers[-1] - mids[-1])
    return np.concatenate([[first], mids, [last]])


def plot_period_comparison(period_results: dict[str, dict], save_path: Path):
    names = list(period_results.keys())
    log_values = []
    for result in period_results.values():
        data = result['period_days'][:, 1:]
        valid = data[np.isfinite(data) & (data > 0)]
        if valid.size:
            log_values.append(np.log10(valid))
    if not log_values:
        raise ValueError('No finite period values found for plotting.')
    log_all = np.concatenate(log_values)
    vmin, vmax = np.quantile(log_all, [0.02, 0.98])

    fig, axes = plt.subplots(len(names), 1, figsize=(12.0, 4.8 * len(names)), sharex=False, sharey=False)
    axes = np.atleast_1d(axes)
    im = None
    for ax, name in zip(axes, names):
        result = period_results[name]
        heat = np.ma.masked_invalid(np.log10(result['period_days'][:, 1:]))
        im = ax.imshow(heat, aspect='auto', cmap='viridis', vmin=vmin, vmax=vmax)
        num_dims = result['period_days'].shape[1]
        xticks = np.arange(0, num_dims - 1, max(1, (num_dims - 1) // 16))
        ax.set_xticks(xticks)
        ax.set_xticklabels([f'dim_{dim_idx + 1:03d}' for dim_idx in xticks], rotation=45, ha='right')
        ax.set_yticks(np.arange(result['period_days'].shape[0]))
        ax.set_yticklabels([f'head {head_idx}' for head_idx in range(result['period_days'].shape[0])])
        ax.set_title(name, fontsize=13)
        ax.set_xlabel('periodic embedding dimension')
        ax.set_ylabel('attention head')

    fig.suptitle('mTAN time-embedding periods: ALBEF vs Optical-only', fontsize=15, y=0.985)
    fig.subplots_adjust(top=0.90, bottom=0.22, hspace=0.52)
    cb = fig.colorbar(im, ax=axes.tolist(), orientation='horizontal', fraction=0.035, pad=0.18)
    cb.set_label('log10(period [days])')
    fig.savefig(save_path, dpi=180, bbox_inches='tight')
    return fig


PERIOD_RESULTS = {name: compute_period_result(bundle) for name, bundle in MODEL_BUNDLES.items()}

period_output_manifest = []
period_summary_rows = []
for name, result in PERIOD_RESULTS.items():
    slug = slugify(name)
    long_csv = OUTPUT_DIR / f'{slug}_mtan_time_embedding_periods_long.csv'
    matrix_csv = OUTPUT_DIR / f'{slug}_mtan_time_embedding_periods_matrix_days.csv'
    result['period_long_df'].to_csv(long_csv, index=False)
    result['period_matrix_df'].to_csv(matrix_csv)
    finite = result['period_days'][:, 1:]
    finite = finite[np.isfinite(finite)]
    period_output_manifest.append({'model': name, 'long_csv': str(long_csv), 'matrix_csv': str(matrix_csv)})
    period_summary_rows.append({
        'model': name,
        'time_embedding_prefix': result['prefix'],
        'period_days_min': float(finite.min()),
        'period_days_median': float(np.median(finite)),
        'period_days_p90': float(np.quantile(finite, 0.9)),
        'period_days_max': float(finite.max()),
    })

period_compare_png = OUTPUT_DIR / 'mtan_time_embedding_periods_albef_vs_optical_only.png'
fig = plot_period_comparison(PERIOD_RESULTS, period_compare_png)
display(pd.DataFrame(period_summary_rows))
plt.show()
print({'period_compare_png': str(period_compare_png), 'period_outputs': period_output_manifest})

## Shared Light-Curve Attention Example

The next cells load the same KN training sample from the optical-only training HDF5 and compare how the two optical encoders distribute time attention after averaging over the same five offset days.


In [ ]:
def luptitude_to_flux_and_sigma(values_lupt: np.ndarray, errors_lupt: np.ndarray, band_index: np.ndarray, lupt_b_njy: np.ndarray):
    b = lupt_b_njy[band_index]
    two_b = 2.0 * b
    x = (PSFFLUX_ZP - values_lupt) / ASINH_MAG_FACTOR - np.log(b)
    flux = two_b * np.sinh(x)
    fluxerr = np.abs(errors_lupt) * np.sqrt(np.square(flux) + np.square(two_b)) / ASINH_MAG_FACTOR
    return flux, fluxerr


def flux_to_mag(flux_njy: np.ndarray, fluxerr_njy: np.ndarray):
    valid = np.isfinite(flux_njy) & np.isfinite(fluxerr_njy) & (flux_njy > 0) & (fluxerr_njy > 0)
    mag = np.full_like(flux_njy, np.nan, dtype=np.float64)
    magerr = np.full_like(flux_njy, np.nan, dtype=np.float64)
    if np.any(valid):
        mag[valid] = PSFFLUX_ZP - 2.5 * np.log10(flux_njy[valid])
        magerr[valid] = (2.5 / np.log(10.0)) * (fluxerr_njy[valid] / flux_njy[valid])
    return mag, magerr, valid


def plot_status_errorbars(ax, x, y, yerr, is_detection, color):
    is_detection = np.asarray(is_detection, dtype=bool)
    if np.any(is_detection):
        ax.errorbar(x[is_detection], y[is_detection], yerr=yerr[is_detection], fmt='o', markersize=4.4, color=color, ecolor=color, elinewidth=1.0, capsize=0, alpha=0.95)
    if np.any(~is_detection):
        ax.errorbar(x[~is_detection], y[~is_detection], yerr=yerr[~is_detection], fmt='s', markersize=4.0, mfc='white', mec=color, color=color, ecolor=color, elinewidth=0.9, capsize=0, alpha=0.82)


def load_example_sample(h5_path: Path, sample_index: int, time_window_scaled: tuple[float, float]):
    with h5py.File(h5_path, 'r') as f:
        grp = f['events/optical_data']
        n_samples = int(grp['times'].shape[0])
        if not (0 <= int(sample_index) < n_samples):
            raise IndexError(f'sample_index={sample_index} is outside [0, {n_samples})')
        raw = {
            'sample_index': int(sample_index),
            'times': np.asarray(grp['times'][sample_index], dtype=np.float32),
            'values': np.asarray(grp['values'][sample_index], dtype=np.float32),
            'errors': np.asarray(grp['errors'][sample_index], dtype=np.float32),
            'masks': np.asarray(grp['masks'][sample_index], dtype=np.float32),
            'coordinates': np.asarray(grp['coordinates'][sample_index], dtype=np.float32),
            'zero_time_mjd_base': float(np.asarray(grp['zero_time_mjd_base'][sample_index], dtype=np.float64)),
            'lupt_b_njy': np.asarray(f.attrs['lupt_b_njy'], dtype=np.float64),
            'band_order': [b.strip().lower() for b in str(f.attrs.get('lupt_band_order', 'u,g,r,i,z,Y')).split(',')],
        }
        for key in ('slot_is_detection', 'meta_n_det', 'meta_n_bands', 'meta_n_obs', 'meta_t_span'):
            if key in grp:
                value = np.asarray(grp[key][sample_index])
                raw[key] = value.item() if value.ndim == 0 else value.astype(np.float32)

    start_scaled, end_scaled = map(float, time_window_scaled)
    slot_valid_full = raw['masks'].sum(axis=1) > 0
    in_window = np.isfinite(raw['times']) & (raw['times'] >= start_scaled) & (raw['times'] <= end_scaled)
    keep_mask = slot_valid_full & in_window
    if not np.any(keep_mask):
        raise RuntimeError(f'sample_index={sample_index} has no observations inside window={time_window_scaled}')

    cropped = dict(raw)
    cropped['n_points_full'] = int(slot_valid_full.sum())
    cropped['n_points_window'] = int(keep_mask.sum())
    cropped['time_window_scaled'] = tuple(float(x) for x in time_window_scaled)
    cropped['time_window_days'] = tuple(float(x * TIME_SCALE_DAYS) for x in time_window_scaled)
    for key in ('times', 'values', 'errors', 'masks'):
        cropped[key] = np.asarray(raw[key])[keep_mask]
    cropped['slot_is_detection'] = np.asarray(raw.get('slot_is_detection', np.ones_like(raw['times'])), dtype=np.float32)[keep_mask]
    cropped['n_det_window'] = int(np.count_nonzero(cropped['slot_is_detection'] > 0.5))
    cropped['n_bands_window'] = int(np.count_nonzero(cropped['masks'].sum(axis=0) > 0))
    return cropped


def make_lightcurve_dataframe(sample: dict) -> pd.DataFrame:
    slot_valid = sample['masks'].sum(axis=1) > 0
    slot_indices = np.flatnonzero(slot_valid)
    rows = []
    for slot_idx in slot_indices:
        band_idx = int(np.argmax(sample['masks'][slot_idx]))
        band = sample['band_order'][band_idx]
        t_scaled = float(sample['times'][slot_idx])
        mjd = sample['zero_time_mjd_base'] + TIME_SCALE_DAYS * t_scaled
        lupt = float(sample['values'][slot_idx, band_idx])
        lupt_err = float(abs(sample['errors'][slot_idx, band_idx]))
        flux_njy, fluxerr_njy = luptitude_to_flux_and_sigma(
            np.asarray([lupt], dtype=np.float64),
            np.asarray([lupt_err], dtype=np.float64),
            np.asarray([band_idx], dtype=np.int64),
            sample['lupt_b_njy'],
        )
        mag, magerr, mag_valid = flux_to_mag(flux_njy, fluxerr_njy)
        rows.append({
            'slot_idx': int(slot_idx),
            'band': band,
            't_days': t_scaled * TIME_SCALE_DAYS,
            'mjd': mjd,
            'lupt': lupt,
            'lupt_err': lupt_err,
            'mag': float(mag[0]) if mag_valid[0] else float('nan'),
            'magerr': float(magerr[0]) if mag_valid[0] else float('nan'),
            'is_detection': bool(sample['slot_is_detection'][slot_idx] > 0.5),
        })
    return pd.DataFrame(rows)


def compute_time_attention(bundle: dict, sample: dict, offset_days: np.ndarray) -> dict:
    curve_encoder = bundle['curve_encoder']
    curve_encoder.eval()

    opt_t = torch.from_numpy(sample['times'][None, :]).to(device=device, dtype=torch.float32)
    opt_v = torch.from_numpy(sample['values'][None, :, :]).to(device=device, dtype=torch.float32)
    opt_mask = torch.from_numpy(sample['masks'][None, :, :]).to(device=device, dtype=torch.float32)
    opt_err = torch.from_numpy(sample['errors'][None, :, :]).to(device=device, dtype=torch.float32)
    ref_time = build_ref_time(1, bundle['n_ref'], bundle['ref_start'], bundle['ref_end'], device, opt_t.dtype)

    slot_valid_t = (opt_mask.sum(dim=-1) > 0)
    band_present = (opt_mask.sum(dim=1) > 0)
    shifted_t_stack = []
    attn_norm_stack = []

    for off_days in np.asarray(offset_days, dtype=np.float64).tolist():
        delta_days = torch.full((1,), float(off_days), device=device, dtype=torch.float32)
        shifted_t = apply_time_offsets(opt_t, opt_mask, delta_days=delta_days, scale_divisor=bundle['offset_scale_days_divisor'])
        amp_ctx = autocast(device_type='cuda', dtype=torch.bfloat16, enabled=(device.type == 'cuda')) if device.type == 'cuda' else nullcontext()
        with torch.no_grad():
            with amp_ctx:
                _, _, attn_weights = curve_encoder(shifted_t, opt_v, ref_time, opt_mask, errors_obs=opt_err, return_attn=True)
        attn_ref = attn_weights.float()[:, :, 1:, :, :]
        attn_head_mean = attn_ref.mean(dim=1)
        band_present_f = band_present[:, None, None, :].to(dtype=attn_head_mean.dtype)
        attn_band_sum = (attn_head_mean * band_present_f).sum(dim=-1)
        slot_valid_f = slot_valid_t[:, None, :].to(dtype=attn_band_sum.dtype)
        attn_band_sum = attn_band_sum * slot_valid_f
        row_sum = attn_band_sum.sum(dim=-1, keepdim=True)
        attn_norm = torch.where(row_sum > 0, attn_band_sum / row_sum, torch.zeros_like(attn_band_sum))
        shifted_t_stack.append(shifted_t.detach().cpu().numpy()[0])
        attn_norm_stack.append(attn_norm.detach().cpu().numpy()[0])

    shifted_t_stack = np.asarray(shifted_t_stack, dtype=np.float64)
    attn_norm_stack = np.asarray(attn_norm_stack, dtype=np.float64)
    attn_mean = attn_norm_stack.mean(axis=0)
    shifted_mean = shifted_t_stack.mean(axis=0)
    slot_valid_np = sample['masks'].sum(axis=1) > 0
    slot_band_idx = np.argmax(sample['masks'][slot_valid_np], axis=1)
    idx_to_band = {i: b for i, b in enumerate(sample['band_order'])}
    slot_band_labels = np.asarray([idx_to_band[int(i)] for i in slot_band_idx], dtype=object)
    heat = attn_mean[:, slot_valid_np]
    obs_time_mean_days = shifted_mean[slot_valid_np] * TIME_SCALE_DAYS
    ref_axis_days = ref_time[0].detach().cpu().numpy().astype(np.float64) * TIME_SCALE_DAYS
    return {
        'name': bundle['name'],
        'heat': heat,
        'ref_axis_days': ref_axis_days,
        'obs_time_mean_days': obs_time_mean_days,
        'slot_band_labels': slot_band_labels,
        'offset_days': np.asarray(offset_days, dtype=np.float64),
    }


In [ ]:
OFFSET_DAYS = load_offset_days(OFFSET_NPZ, OFFSET_KEY, OFFSET_QUANTILES)
EXAMPLE_SAMPLE = load_example_sample(TRAIN_H5, EXAMPLE_SAMPLE_INDEX, COMMON_WINDOW_SCALED)
LC_DF = make_lightcurve_dataframe(EXAMPLE_SAMPLE)
ATTENTION_RESULTS = {name: compute_time_attention(bundle, EXAMPLE_SAMPLE, OFFSET_DAYS) for name, bundle in MODEL_BUNDLES.items()}

print(json.dumps({
    'sample_index': EXAMPLE_SAMPLE['sample_index'],
    'time_window_days': EXAMPLE_SAMPLE['time_window_days'],
    'n_points_window': EXAMPLE_SAMPLE['n_points_window'],
    'n_det_window': EXAMPLE_SAMPLE['n_det_window'],
    'n_bands_window': EXAMPLE_SAMPLE['n_bands_window'],
    'offset_days': [float(x) for x in OFFSET_DAYS.tolist()],
}, indent=2))
display(LC_DF.head(12))


In [ ]:
def plot_attention_comparison(lightcurve_df: pd.DataFrame, attention_results: dict[str, dict], sample: dict, save_path: Path):
    names = list(attention_results.keys())
    vmax = max(float(np.nanpercentile(result['heat'], 99)) for result in attention_results.values())
    vmax = max(vmax, 1.0e-6)

    fig = plt.figure(figsize=(8.0 * len(names), 11.0))
    gs = fig.add_gridspec(3, len(names), height_ratios=[1.0, 1.0, 1.45], hspace=0.18, wspace=0.16)
    ax_lupt = fig.add_subplot(gs[0, :])
    ax_mag = fig.add_subplot(gs[1, :], sharex=ax_lupt)
    heat_axes = [fig.add_subplot(gs[2, col]) for col in range(len(names))]

    for band, grp in lightcurve_df.groupby('band', sort=False):
        color = BAND_COLORS.get(str(band).lower(), '#333333')
        x = grp['mjd'].to_numpy(dtype=np.float64)
        plot_status_errorbars(ax_lupt, x, grp['lupt'].to_numpy(dtype=np.float64), grp['lupt_err'].to_numpy(dtype=np.float64), grp['is_detection'].to_numpy(dtype=bool), color)
        mag_valid = np.isfinite(grp['mag'].to_numpy(dtype=np.float64)) & np.isfinite(grp['magerr'].to_numpy(dtype=np.float64))
        if np.any(mag_valid):
            sub = grp.loc[mag_valid]
            plot_status_errorbars(ax_mag, sub['mjd'].to_numpy(dtype=np.float64), sub['mag'].to_numpy(dtype=np.float64), sub['magerr'].to_numpy(dtype=np.float64), sub['is_detection'].to_numpy(dtype=bool), color)

    window_start_days, window_end_days = sample['time_window_days']
    ax_lupt.set_title(
        f'Shared training KN sample | idx={sample["sample_index"]} | window=[{window_start_days:+.1f}, {window_end_days:+.1f}] d | '
        f'n_det={sample["n_det_window"]} | n_bands={sample["n_bands_window"]}'
    )
    ax_lupt.set_ylabel('luptitude')
    ax_lupt.grid(alpha=0.25, linestyle='--')
    ax_lupt.invert_yaxis()
    ax_mag.set_xlabel('MJD')
    ax_mag.set_ylabel('AB magnitude\n(positive flux only)')
    ax_mag.grid(alpha=0.25, linestyle='--')
    ax_mag.invert_yaxis()

    band_handles = [Line2D([0], [0], color=BAND_COLORS[b], marker='o', linestyle='None', label=b) for b in BANDS]
    status_handles = [
        Line2D([0], [0], color='#333333', marker='o', linestyle='None', label='detection'),
        Line2D([0], [0], color='#333333', marker='s', mfc='white', linestyle='None', label='non-detection / forced'),
    ]
    ax_lupt.legend(handles=band_handles + status_handles, loc='center left', bbox_to_anchor=(1.01, 0.5), frameon=False)

    mesh = None
    for ax, name in zip(heat_axes, names):
        result = attention_results[name]
        heat = result['heat']
        ref_axis_days = result['ref_axis_days']
        obs_time_mean_days = result['obs_time_mean_days']
        slot_band_labels = result['slot_band_labels']
        obs_point_idx = np.arange(1, heat.shape[1] + 1, dtype=np.float64)
        obs_edges = np.arange(0.5, heat.shape[1] + 1.5, 1.0, dtype=np.float64)
        ref_edges = centers_to_edges(ref_axis_days, default_half_width=0.5 * (TIME_SCALE_DAYS / max(1, len(ref_axis_days) - 1)))
        mesh = ax.pcolormesh(obs_edges, ref_edges, heat, shading='auto', cmap='magma', vmin=0.0, vmax=vmax)
        ax.set_title(f'{name} | mean over heads, {len(result["offset_days"])} offsets')
        ax.set_xlabel('observation point + band')
        ax.set_xticks(obs_point_idx)
        ax.set_xticklabels(
            [f'{idx}-{band}\n{time_mean:+.2f}d' for idx, band, time_mean in zip(obs_point_idx.astype(int), slot_band_labels, obs_time_mean_days)],
            fontsize=8,
        )
        ax.tick_params(axis='x', rotation=0)
        ax.set_xlim(0.5, heat.shape[1] + 0.5)
        ax.set_xticks(obs_edges, minor=True)
        ax.grid(which='minor', axis='x', alpha=0.18, linestyle='-', linewidth=0.6)
        ax.grid(which='major', axis='y', alpha=0.12, linestyle='--', linewidth=0.7)
    heat_axes[0].set_ylabel('reference time [days]')
    for ax in heat_axes[1:]:
        ax.set_ylabel('')
    cb = fig.colorbar(mesh, ax=heat_axes, fraction=0.025, pad=0.02)
    cb.set_label('attention weight (row-normalized over observed slots)')
    fig.suptitle('Time attention comparison on the same KN light curve', fontsize=15)
    fig.tight_layout(rect=[0, 0, 0.95, 0.96])
    fig.savefig(save_path, dpi=180, bbox_inches='tight')
    return fig


attention_compare_png = OUTPUT_DIR / 'training_kn_attention_albef_vs_optical_only.png'
attention_meta_json = OUTPUT_DIR / 'training_kn_attention_albef_vs_optical_only.json'
fig = plot_attention_comparison(LC_DF, ATTENTION_RESULTS, EXAMPLE_SAMPLE, attention_compare_png)
attention_meta = {
    'sample_index': int(EXAMPLE_SAMPLE['sample_index']),
    'time_window_days': [float(x) for x in EXAMPLE_SAMPLE['time_window_days']],
    'offset_days': [float(x) for x in OFFSET_DAYS.tolist()],
    'models': {name: {'checkpoint': str(bundle['ckpt_path']), 'epoch': bundle['epoch']} for name, bundle in MODEL_BUNDLES.items()},
    'output_png': str(attention_compare_png),
}
attention_meta_json.write_text(json.dumps(attention_meta, indent=2), encoding='utf-8')
plt.show()
print(json.dumps(attention_meta, indent=2))


## Notes

- To compare different checkpoints, edit `ALBEF_CKPT` and `OPTICAL_ONLY_CKPT` in the config cell and rerun from the model-loading cell onward.
- To inspect a different KN example, change `EXAMPLE_SAMPLE_INDEX` and rerun from the sample-loading cell onward.
- The attention comparison intentionally uses the same optical-only training sample for both encoders so the bottom-row heatmaps isolate encoder behavior.
